# Paloma Humans Datasets

Create:
- Make European-only human HA-only pandemic swine dataset

Trees: Make 2 Fast Trees
- One with all pig sequences + human dataset
- One with only Paloma sequences + human dataset

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [2]:
# Directory paths

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/"
downloads = home + "downloads/human_euro_pandemic_h1n1_ha_2009--2026/" 
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

complete_files = home + "complete_human/" 
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# if not os.path.exists(complete_files + "deduplicated/"): # checking if the directory exists or not
#     os.makedirs(complete_files + "deduplicated/") # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

In [3]:
# Get metadata and sequences

gisaid_metadata = []
gisaid_fastas = []
for dirpath, dirs, files in os.walk(downloads):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if ".fasta" in file_name:
            fasta = fasta_df(file_name, states_ref)
            # fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))
            # # All sequences should be human
            # fasta_df["Isolate"] = fasta_df["full_header"].apply(lambda x: x.split("/")[2]) # if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
            # fasta_df["Subtype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-5])
            # fasta_df["Geo_Location"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-4])
            # fasta_df["Collection_Date"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-3])
            # fasta_df["Host_Type"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-2])
            # fasta_df["Genotype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-1])
            gisaid_fastas.append(fasta)
        else: # if ".xls" in file name
            metadata = pd.read_excel(file_name)
            gisaid_metadata.append(metadata)

# Concatenate metadata
metadata_concat = pd.DataFrame()
for metadata_file in gisaid_metadata:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

# Concatenate fastas
fasta_concat = pd.DataFrame()
for fasta in gisaid_fastas:
    fasta_concat = pd.concat([fasta_concat, fasta])

print(metadata_concat)
print(fasta_concat)


EPI_ISL_20245114|A_Arkhangelsk_63_CRIE_2025|A_/_H1N1|HA|2025-10-17
             Isolate_Id                            PB2 Segment_Id  \
0        EPI_ISL_100110                                       NaN   
1        EPI_ISL_230445        EPI815062|3000479555_N8K8H66E_v1_1   
2         EPI_ISL_65700                                       NaN   
3        EPI_ISL_164087               EPI535791|A/Paris/5236/2009   
4        EPI_ISL_263066                                       NaN   
...                 ...                                       ...   
16519  EPI_ISL_19855954  EPI4308146|MH233195_S358_R1_001_PB2_cons   
16520  EPI_ISL_19855953  EPI4308138|MH233194_S346_R1_001_PB2_cons   
16521  EPI_ISL_19855952  EPI4308130|MH233193_S334_R1_001_PB2_cons   
16522  EPI_ISL_19855950  EPI4308114|MH233177_S310_R1_001_PB2_cons   
16523  EPI_ISL_19855949  EPI4308106|MH233176_S298_R1_001_PB2_cons   

                                 PB1 Segment_Id  \
0                                           NaN   
1 

## Make names

In [ ]:
# Find "animals" (geographic locations indicating human sequence)

segment_fastas = []
unique_animals_all = []

unique_animals = sort_animals(fasta_concat) # Find unique animals
    # print("Animals: ", unique_animals)
unique_animals_all.append(unique_animals)

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "pet_food", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(references)
animals_df.to_csv("animals_ref_to_sort.csv")

['merthyrtydfil', 'eskilstuna', 'portalegre_pt', 'conwy', 'chalkida_gr', 'alania', 'gornoye_veretye', 'serres', 'kosh-agach', 'majkop', 'siena', 'lubica', 'iceland', 'dambovita', 'lipesk', 'chester', 'gavle', 'kharkiv', 'st._petersburg', 'tessenderlo', 'abidjan', 'llandybie', 'quimper', 'mold', 'trencin', 'lesvos.gr', 'nizhny_novgorod_oblast', 'larisa', 'bolzano', 'milfordhaven', 'mshaga', 'sarn', 'belfast', 'andalucia', 'minera', 'fvg-gorizia', 'tyumen', 'briton_ferry', 'lisboa134', 'desgenettes', 'cwmdare', 'irakleio.gr', 'toulon', 'trieste', 'romania_ms', 'llantrisant', 'yekaterinburg', 'bosnia_&_herzegovina', 'kursk', 'burga', 'krasnoyarsk', 'gibraltar', 'baranocihi', 'volgograd_oblast', 'archangelsk', 'blagoveshensk', 'pontllanfraith', 'markham', 'ruthin', 'malacky', 'bashkortostan', 'bojnice', 'cheboksary', 'athens_gr', 'aquitaine', 'north_ossetia', 'odense', 'constanta', 'livadia.gr', 'wurzburg', 'manorbier', 'merthyr_tydfil', 'kerkira', 'thessaloniki', 'annecy', 'ostrava_221', 

In [15]:
fasta = fix_animals(fasta_concat, animals_df) # Fix animals first

# >EPI_ID|Isolate_name|subtype|collection_date|host_type

xls = metadata_concat.rename(columns={"Isolate_Id":"Identifier"})

# Merge metadata with fasta
fasta_meta = fasta_concat.merge(xls, how="right", on="Identifier")

print(fasta_meta)
# Those with missing metadata get dropped
fasta_meta = fasta_meta.dropna(subset=["Identifier", "Isolate_Name_x", "Subtype_x", "Geo_Location", "Date Collected", "Host_Type"])

# Rename sequences 
new_name = ">" + fasta_meta["Identifier"] + "|" + fasta_meta["Isolate_Name_x"] + "|" + fasta_meta["Subtype_x"] + "|" + fasta_meta["Geo_Location"] + "|" + fasta_meta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_meta["Host_Type"] 
# print(fasta_seg["New_Name"])

fasta_meta["full_header"] = new_name

print(fasta_meta)


                                                  Header     Isolate_Id  \
0      EPI_ISL_100110|A/Austria/104/2011|A_/_H1N1|HA|...            104   
1      EPI_ISL_230445|A/Khmelnitsky/667/2016|A_/_H1N1...            667   
2      EPI_ISL_65700|A/Italy/215/2009|A_/_H1N1|HA|200...            215   
3      EPI_ISL_164087|A/Paris/5236/2009|A_/_H1N1|HA|2...           5236   
4      EPI_ISL_263066|A/Sachsen/41/2017|A_/_H1N1|HA|2...             41   
...                                                  ...            ...   
62277  EPI_ISL_19855954|A/Vologda/RII-MH233195S/2025|...  RII-MH233195S   
62278  EPI_ISL_19855953|A/Vologda/RII-MH233194S/2025|...  RII-MH233194S   
62279  EPI_ISL_19855952|A/Vologda/RII-MH233193S/2025|...  RII-MH233193S   
62280  EPI_ISL_19855950|A/Vologda/RII-MH233177S/2025|...  RII-MH233177S   
62281  EPI_ISL_19855949|A/Vologda/RII-MH233176S/2025|...  RII-MH233176S   

                     Isolate_Name_x Subtype_x Segment Location_Header  \
0                A/Austria

## Create FASTA

In [18]:
fasta_meta = fasta_meta.rename(columns={"Sequence":"sequence"})
df_to_fasta(fasta_meta, "pandemic_H1_HA_euro_human_2009-01-01--2026-02-05.fasta", complete_files)